In [ ]:
# Load R magic extension for Python Jupyter kernel (RData loading)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# 5-Class Emergency Department Triage Classifier: FedMML Training & External 5V Generalization Benchmark (`models/train_fedmm_classifier.ipynb`)

This notebook trains **LightGBM Emergency Severity Index (ESI 1..5) triage models** on the **Federated Multi-Modal Emergency Dataset (`datasets/fedmml_ed_triage_dataset.csv`)** and subsequently evaluates an **External Zero-Shot Generalization Benchmark on the 5-Variable Dataset (`datasets/5v_cleandf.RData`)**:

### 📊 Feature Roster & Cross-Dataset Translation
| Feature / Target | FedMML Dataset (`fedmml_ed_triage_dataset.csv`) | 5V Dataset (`5v_cleandf.RData`) | Mapping / Encoding Logic |
| :--- | :--- | :--- | :--- |
| **Age** | `age` | `age` | Continuous numerical (years) |
| **Sex / Gender** | `sex` (`'M'`, `'F'`) | `gender` (`'Male'`, `'Female'`) | Binary integer: `1 = Male`, `0 = Female` |
| **Systolic BP** | `systolic_bp` | `triage_vital_sbp` | Continuous numerical (mmHg) |
| **Heart Rate** | `heart_rate` | `triage_vital_hr` | Continuous numerical (bpm) |
| **Respiratory Rate** | `respiratory_rate` | `triage_vital_rr` | Continuous numerical (breaths/min) |
| **Oxygen Saturation** | `spo2` | `triage_vital_o2` | Continuous numerical (%) |
| **Target Class** | `esi_level` (`1, 2, 3, 4, 5`) | `esi` (`'1', '2', '3', '4', '5'`) | Discrete 5-class target ($1..5$) |

### 🧹 Complete Case Cleaning Rules
- **FedMML Dataset**: Strictly drops any row containing $\ge 1$ null value in `['age', 'sex', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2', 'esi_level']` (retains **81,742 complete cases**).
- **External 5V Dataset**: Strictly skips and drops any row containing $\ge 1$ null value in `['age', 'gender', 'triage_vital_sbp', 'triage_vital_hr', 'triage_vital_rr', 'triage_vital_o2', 'esi']` (retains **282,558 complete cases**).

```mermaid
flowchart TD
    FedMML["FedMML Dataset (81,742 complete rows)"] --> Split["Stratified 3-Way Split via config/triage_conf.json (Train 70%, Val 20%, Test 10%)"]
    Split --> Preproc["StandardScaler Normalization"]
    
    Preproc --> ModelA["Model A: Direct 5-Class LightGBM (objective='multiclass')"]
    Preproc --> ModelB["Model B: Hierarchical 4-Tier LightGBM Stacking + LogReg Meta"]
    
    ModelA & ModelB --> TestInternal["Internal Benchmark: FedMML Holdout Test Set (8,175 rows)"]
    
    Dataset5V["External 5V Dataset (282,558 complete rows)"] --> Map5V["Feature Translation & Scaling"]
    Map5V --> TestExternal["External Generalization Benchmark: 5V Dataset (282,558 rows)"]
    
    TestInternal & TestExternal --> Comparison["Cross-Dataset Performance Comparison & Diagnostic Visualizations"]
```

In [ ]:
# ---------------------------------------------------------
# Step 1: Load FedMML Dataset, Filter Nulls, Encode 'sex', & Stratified 3-Way Split via triage_conf.json
# ---------------------------------------------------------
import os, json, pickle, time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, recall_score,
    precision_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

# 1. Load Partitioning Configuration
config_path = f"{ROOT}/config/triage_conf.json"
with open(config_path, 'r') as f:
    config = json.load(f)

test_size = config['training']['test_size']
val_size  = config['training']['val_size']
seed_val  = config['training']['random_state']

print(f"Loaded Configuration from {config_path}:")
print(f"  * Test Size Fraction       = {test_size:.2f} ({test_size*100:.1f}%)")
print(f"  * Validation Size Fraction = {val_size:.2f} ({val_size*100:.1f}%)")
print(f"  * Random State Seed        = {seed_val}")
print("-" * 80)

# 2. Load Raw FedMML Dataset
data_path = f"{ROOT}/datasets/fedmml_ed_triage_dataset.csv"
print(f"Loading FedMML dataset from: {data_path}...")
raw_df = pd.read_csv(data_path)
initial_rows = len(raw_df)

required_features = ['age', 'sex', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2']
target_col = 'esi_level'
all_required_cols = required_features + [target_col]

print("=" * 80)
print(f"  FEDMML DATASET: {initial_rows:,} Total Rows, {raw_df.shape[1]} Columns")
print("=" * 80)
print("Null counts per required column before filtering:")
print(raw_df[all_required_cols].isnull().sum())
print("-" * 80)

# 3. Strictly Drop Rows With >= 1 Null in Required Features or Target
clean_df = raw_df.dropna(subset=all_required_cols).copy()
clean_rows = len(clean_df)
dropped_rows = initial_rows - clean_rows

print(f"✓ Dropped {dropped_rows:,} rows with missing values (Retained {clean_rows:,} complete cases, {clean_rows/initial_rows*100:.2f}%)")

# 4. Encode 'sex' Feature (M -> 1, F -> 0)
clean_df['sex_encoded'] = clean_df['sex'].astype(str).str.strip().str.upper().map({'M': 1, 'MALE': 1, 'F': 0, 'FEMALE': 0})
clean_df = clean_df.dropna(subset=['sex_encoded']).copy()
clean_df['sex_encoded'] = clean_df['sex_encoded'].astype(int)
clean_df[target_col] = clean_df[target_col].astype(int)

feature_names = ['age', 'sex_encoded', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2']

print("\nCleaned FedMML Target Distribution ('esi_level'):")
esi_dist = clean_df[target_col].value_counts().sort_index()
for cls_val, cnt in esi_dist.items():
    print(f"  * ESI Level {cls_val} : {cnt:,} cases ({cnt/len(clean_df)*100:.2f}%)")
print("=" * 80)

# 5. Stratified 3-Way Partitioning based on config/triage_conf.json
X_all = clean_df[feature_names].values
y_all = clean_df[target_col].values

# (a) Extract Stratified Holdout Test Set
X_rem_raw, X_test_raw, y_rem, y_test = train_test_split(
    X_all, y_all, test_size=test_size, stratify=y_all, random_state=seed_val
)

# (b) Extract Stratified Validation Set from remainder
val_adj_fraction = val_size / (1.0 - test_size)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_rem_raw, y_rem, test_size=val_adj_fraction, stratify=y_rem, random_state=seed_val + 1
)

print(f"Stratified Partition Complete (Referenced from triage_conf.json):")
print(f"  * Train Set      : {len(X_train_raw):,} rows ({len(X_train_raw)/len(clean_df)*100:.1f}%)")
print(f"  * Validation Set : {len(X_val_raw):,} rows ({len(X_val_raw)/len(clean_df)*100:.1f}%)")
print(f"  * Holdout Test   : {len(X_test_raw):,} rows ({len(X_test_raw)/len(clean_df)*100:.1f}%)")

In [ ]:
# ---------------------------------------------------------
# Step 2: Feature Matrix Normalization (StandardScaler on Continuous Vitals)
# ---------------------------------------------------------
cont_indices = [0, 2, 3, 4, 5]  # age, systolic_bp, heart_rate, respiratory_rate, spo2

scaler = StandardScaler()
X_train = X_train_raw.copy()
X_val   = X_val_raw.copy()
X_test  = X_test_raw.copy()

X_train[:, cont_indices] = scaler.fit_transform(X_train_raw[:, cont_indices])
X_val[:, cont_indices]   = scaler.transform(X_val_raw[:, cont_indices])
X_test[:, cont_indices]  = scaler.transform(X_test_raw[:, cont_indices])

print(f"✓ Feature Matrices Normalized: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

In [ ]:
# ---------------------------------------------------------
# Step 3: Train Model A (Direct Multiclass) & Model B (Hierarchical Stacking)
# ---------------------------------------------------------
print("=" * 80)
print("  TRAINING MODEL A: DIRECT 5-CLASS MULTI-CLASS LIGHTGBM CLASSIFIER")
print("=" * 80)

lgb_mc_params = {
    'objective': 'multiclass',
    'num_class': 5,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'class_weight': 'balanced',
    'feature_fraction': 0.85,
    'bagging_fraction': 0.85,
    'bagging_freq': 1,
    'min_child_samples': 20,
    'n_estimators': 200,
    'verbosity': -1,
    'random_state': 42
}

t0 = time.time()
model_mc_lgbm = lgb.LGBMClassifier(**lgb_mc_params)
model_mc_lgbm.fit(
    X_train, y_train - 1,
    eval_set=[(X_val, y_val - 1)],
    callbacks=[lgb.early_stopping(stopping_rounds=15, verbose=False)]
)
print(f"✓ Direct 5-Class LightGBM trained in {time.time()-t0:.1f}s!")

print("\n" + "=" * 80)
print("  TRAINING MODEL B: HIERARCHICAL 4-TIER LIGHTGBM STACKING PIPELINE")
print("=" * 80)

def binary_numpy_smote(X, y_bin, seed=42):
    np.random.seed(seed)
    pos_mask = (y_bin == 1)
    neg_mask = (y_bin == 0)
    n_pos = np.sum(pos_mask)
    n_neg = np.sum(neg_mask)
    if n_pos == 0 or n_neg == 0 or n_pos == n_neg:
        return X, y_bin
    if n_pos < n_neg:
        min_X = X[pos_mask]; target_syn = n_neg - n_pos; min_label = 1
    else:
        min_X = X[neg_mask]; target_syn = n_pos - n_neg; min_label = 0
    n_min = len(min_X)
    syn_X = np.zeros((target_syn, X.shape[1]), dtype=np.float64)
    for i in range(target_syn):
        idx1 = np.random.randint(0, n_min)
        idx2 = np.random.randint(0, n_min)
        alpha = np.random.rand()
        syn_X[i] = min_X[idx1] + alpha * (min_X[idx2] - min_X[idx1])
    return np.vstack([X, syn_X]), np.hstack([y_bin, np.full(target_syn, min_label)])

lgb_bin_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'feature_fraction': 0.85,
    'bagging_fraction': 0.85,
    'bagging_freq': 1,
    'verbosity': -1,
    'random_state': 42,
    'n_estimators': 150
}

t0 = time.time()
# 1. Layer 1: ESI 1 vs (ESI 2..5) with SMOTE
X_sm1, y_sm1 = binary_numpy_smote(X_train, (y_train == 1).astype(int))
l1_model = lgb.LGBMClassifier(**lgb_bin_params)
l1_model.fit(X_sm1, y_sm1, eval_set=[(X_val, (y_val == 1).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# 2. Layer 2: ESI 2,3 vs ESI 4,5 on Non-ESI 1
m2_tr  = (y_train != 1); m2_val = (y_val != 1)
X_sm2, y_sm2 = binary_numpy_smote(X_train[m2_tr], np.isin(y_train[m2_tr], [2, 3]).astype(int))
l2_model = lgb.LGBMClassifier(**lgb_bin_params)
l2_model.fit(X_sm2, y_sm2, eval_set=[(X_val[m2_val], np.isin(y_val[m2_val], [2, 3]).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# 3. Layer 3A: ESI 2 vs ESI 3 on ESI 2,3
m3a_tr  = np.isin(y_train, [2, 3]); m3a_val = np.isin(y_val, [2, 3])
X_sm3a, y_sm3a = binary_numpy_smote(X_train[m3a_tr], (y_train[m3a_tr] == 2).astype(int))
l3a_model = lgb.LGBMClassifier(**lgb_bin_params)
l3a_model.fit(X_sm3a, y_sm3a, eval_set=[(X_val[m3a_val], (y_val[m3a_val] == 2).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# 4. Layer 3B: ESI 4 vs ESI 5 on ESI 4,5
m3b_tr  = np.isin(y_train, [4, 5]); m3b_val = np.isin(y_val, [4, 5])
X_sm3b, y_sm3b = binary_numpy_smote(X_train[m3b_tr], (y_train[m3b_tr] == 4).astype(int))
l3b_model = lgb.LGBMClassifier(**lgb_bin_params)
l3b_model.fit(X_sm3b, y_sm3b, eval_set=[(X_val[m3b_val], (y_val[m3b_val] == 4).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# Hierarchical Probability Computation
def compute_hierarchical_probs(l1, l2, l3a, l3b, X_in):
    p1  = l1.predict_proba(X_in)[:, 1]
    p2  = l2.predict_proba(X_in)[:, 1]
    p3a = l3a.predict_proba(X_in)[:, 1]
    p3b = l3b.predict_proba(X_in)[:, 1]
    
    P = np.zeros((len(X_in), 5))
    P[:, 0] = p1
    P[:, 1] = (1 - p1) * p2 * p3a
    P[:, 2] = (1 - p1) * p2 * (1 - p3a)
    P[:, 3] = (1 - p1) * (1 - p2) * p3b
    P[:, 4] = (1 - p1) * (1 - p2) * (1 - p3b)
    return P

val_probs_stack  = compute_hierarchical_probs(l1_model, l2_model, l3a_model, l3b_model, X_val)

# Calibrate Multinomial Logistic Regression Meta-Learner on Validation Probabilities
meta_learner = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
meta_learner.fit(val_probs_stack, y_val)

print(f"✓ Hierarchical LightGBM Stacking Pipeline trained & calibrated in {time.time()-t0:.1f}s!")

In [ ]:
# ---------------------------------------------------------
# Step 4: Benchmark 1 — Evaluation on FedMML Holdout Test Set
# ---------------------------------------------------------
def get_per_class_breakdown(y_true, y_pred, probs, pipeline_name, class_list=[1, 2, 3, 4, 5]):
    rows = []
    recalls, specs, bal_accs, precs, f1s, aucs = [], [], [], [], [], []
    for idx, cls in enumerate(class_list):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        f1   = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        try: auc = roc_auc_score(y_bin_true, probs[:, idx])
        except Exception: auc = 0.0
        
        recalls.append(rec); specs.append(spec); bal_accs.append(bal)
        precs.append(prec); f1s.append(f1); aucs.append(auc)
        
        rows.append({
            'Dataset': 'FedMML_Holdout',
            'Pipeline': pipeline_name,
            'Class': f'ESI_{cls}',
            'Recall_Sensitivity': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'Precision_PPV': round(prec, 4),
            'F1_Score': round(f1, 4),
            'ROC_AUC': round(auc, 4)
        })
    rows.append({
        'Dataset': 'FedMML_Holdout',
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall_Sensitivity': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'Precision_PPV': round(np.mean(precs), 4),
        'F1_Score': round(np.mean(f1s), 4),
        'ROC_AUC': round(np.mean(aucs), 4)
    })
    return pd.DataFrame(rows)

# 1. Model A Test Evaluation
probs_test_mc_fedmml = model_mc_lgbm.predict_proba(X_test)
preds_test_mc_fedmml = model_mc_lgbm.predict(X_test) + 1
report_mc_fedmml     = get_per_class_breakdown(y_test, preds_test_mc_fedmml, probs_test_mc_fedmml, 'Direct_Multiclass_LightGBM')

# 2. Model B Test Evaluation
test_probs_stack_fedmml = compute_hierarchical_probs(l1_model, l2_model, l3a_model, l3b_model, X_test)
probs_test_stack_fedmml = meta_learner.predict_proba(test_probs_stack_fedmml)
preds_test_stack_fedmml = meta_learner.predict(test_probs_stack_fedmml)
report_stack_fedmml     = get_per_class_breakdown(y_test, preds_test_stack_fedmml, probs_test_stack_fedmml, 'Hierarchical_Stacking_LightGBM')

print("=" * 115)
print("   [BENCHMARK 1] FEDMML HOLDOUT TEST EVALUATION: DIRECT MULTICLASS LIGHTGBM")
print("=" * 115)
print(report_mc_fedmml[['Class', 'Recall_Sensitivity', 'Specificity', 'Balanced_Accuracy', 'Precision_PPV', 'F1_Score', 'ROC_AUC']].to_string(index=False))
print("=" * 115 + chr(10))

print("=" * 115)
print("   [BENCHMARK 1] FEDMML HOLDOUT TEST EVALUATION: HIERARCHICAL STACKING LIGHTGBM")
print("=" * 115)
print(report_stack_fedmml[['Class', 'Recall_Sensitivity', 'Specificity', 'Balanced_Accuracy', 'Precision_PPV', 'F1_Score', 'ROC_AUC']].to_string(index=False))
print("=" * 115 + chr(10))

reports_dir = f"{ROOT}/reports"
os.makedirs(reports_dir, exist_ok=True)
report_mc_fedmml.to_csv(os.path.join(reports_dir, 'fedmml_internal_direct_multiclass_report.csv'), index=False)
report_stack_fedmml.to_csv(os.path.join(reports_dir, 'fedmml_internal_hierarchical_stacking_report.csv'), index=False)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Load External 5V Dataset (5v_cleandf.RData) & Translate Features
# Strictly skip/drop any row with >= 1 NA feature or NA target class
# ---------------------------------------------------------
data_5v_path <- "../datasets/5v_cleandf.RData"
if (!file.exists(data_5v_path)) data_5v_path <- "datasets/5v_cleandf.RData"

env_5v <- new.env()
load(data_5v_path, envir = env_5v)

df_names_5v <- ls(env_5v)[sapply(ls(env_5v), function(x) is.data.frame(get(x, envir = env_5v)))]
df_sizes_5v <- sapply(df_names_5v, function(x) nrow(get(x, envir = env_5v)))
raw_5v_df   <- get(df_names_5v[which.max(df_sizes_5v)], envir = env_5v)

cat("========================================================================\n")
cat(sprintf("  EXTERNAL 5V DATASET LOADED: %d Total Rows, %d Columns\n", nrow(raw_5v_df), ncol(raw_5v_df)))
cat("========================================================================\n")

# Feature translation to match FedMML format:
# - age              -> age
# - gender           -> sex_encoded (Male: 1, Female: 0)
# - triage_vital_sbp -> systolic_bp
# - triage_vital_hr  -> heart_rate
# - triage_vital_rr  -> respiratory_rate
# - triage_vital_o2  -> spo2
# - esi              -> esi_level (1..5)

gender_num_5v <- ifelse(is.na(raw_5v_df$gender), NA, ifelse(as.character(raw_5v_df$gender) == "Male", 1, 0))
esi_num_5v    <- as.numeric(as.character(raw_5v_df$esi))

df_5v_mapped <- data.frame(
  age              = raw_5v_df$age,
  sex_encoded      = gender_num_5v,
  systolic_bp      = raw_5v_df$triage_vital_sbp,
  heart_rate       = raw_5v_df$triage_vital_hr,
  respiratory_rate = raw_5v_df$triage_vital_rr,
  spo2             = raw_5v_df$triage_vital_o2,
  esi_level        = esi_num_5v
)

cat("Null counts per required feature and target before filtering:\n")
print(sapply(df_5v_mapped, function(x) sum(is.na(x))))
cat("------------------------------------------------------------------------\n")

# Strictly drop any row containing >= 1 null value in the 6 features or target
clean_5v_df <- na.omit(df_5v_mapped)
dropped_5v_rows <- nrow(df_5v_mapped) - nrow(clean_5v_df)

cat(sprintf("5V Complete Case Filtering: Dropped %d rows with >= 1 NA value (Retained %d complete cases, %.2f%%)\n", 
            dropped_5v_rows, nrow(clean_5v_df), (nrow(clean_5v_df) / nrow(raw_5v_df)) * 100))
cat("5V Cleaned Target ESI Distribution (100% complete cases):\n")
print(table(clean_5v_df$esi_level))
cat("========================================================================\n")

export_5v_mat <- as.matrix(clean_5v_df)

In [ ]:
# ---------------------------------------------------------
# Step 6: Benchmark 2 — External Generalization Evaluation on 5V Dataset (5v_cleandf.RData)
# ---------------------------------------------------------
from rpy2.robjects import r

mat_5v = np.array(r('export_5v_mat'), dtype=np.float64)
X_5v_raw = mat_5v[:, :6]   # age, sex_encoded, systolic_bp, heart_rate, respiratory_rate, spo2
y_5v     = mat_5v[:, 6].astype(int)  # ESI 1..5

# Verify zero null/NaN values
assert not np.isnan(X_5v_raw).any(), "Error: Found NaN values in 5V feature matrix!"
assert not np.isnan(y_5v).any(), "Error: Found NaN values in 5V target vector!"

# Normalize continuous columns using FedMML fitted StandardScaler
X_5v = X_5v_raw.copy()
X_5v[:, cont_indices] = scaler.transform(X_5v_raw[:, cont_indices])

print(f"✓ External 5V Test Matrix Ready (100% complete cases): {X_5v.shape}, Classes: {np.unique(y_5v)}")

# 1. Model A Evaluation on External 5V Dataset
probs_5v_mc = model_mc_lgbm.predict_proba(X_5v)
preds_5v_mc = model_mc_lgbm.predict(X_5v) + 1
report_mc_5v = get_per_class_breakdown(y_5v, preds_5v_mc, probs_5v_mc, 'Direct_Multiclass_LightGBM')
report_mc_5v['Dataset'] = 'External_5V_Dataset'

# 2. Model B Evaluation on External 5V Dataset
test_probs_stack_5v = compute_hierarchical_probs(l1_model, l2_model, l3a_model, l3b_model, X_5v)
probs_5v_stack      = meta_learner.predict_proba(test_probs_stack_5v)
preds_5v_stack      = meta_learner.predict(test_probs_stack_5v)
report_stack_5v     = get_per_class_breakdown(y_5v, preds_5v_stack, probs_5v_stack, 'Hierarchical_Stacking_LightGBM')
report_stack_5v['Dataset'] = 'External_5V_Dataset'

print("=" * 115)
print("   [BENCHMARK 2] EXTERNAL 5V GENERALIZATION: DIRECT MULTICLASS LIGHTGBM (N = 282,558)")
print("=" * 115)
print(report_mc_5v[['Class', 'Recall_Sensitivity', 'Specificity', 'Balanced_Accuracy', 'Precision_PPV', 'F1_Score', 'ROC_AUC']].to_string(index=False))
print("=" * 115 + chr(10))

print("=" * 115)
print("   [BENCHMARK 2] EXTERNAL 5V GENERALIZATION: HIERARCHICAL STACKING LIGHTGBM (N = 282,558)")
print("=" * 115)
print(report_stack_5v[['Class', 'Recall_Sensitivity', 'Specificity', 'Balanced_Accuracy', 'Precision_PPV', 'F1_Score', 'ROC_AUC']].to_string(index=False))
print("=" * 115 + chr(10))

# Cross-Dataset Generalization Comparison Table
cross_comp_rows = []
for i in range(len(report_mc_fedmml)):
    cls_lbl = report_mc_fedmml.loc[i, 'Class']
    b_fed_mc = report_mc_fedmml.loc[i, 'Balanced_Accuracy']
    b_5v_mc  = report_mc_5v.loc[i, 'Balanced_Accuracy']
    b_fed_st = report_stack_fedmml.loc[i, 'Balanced_Accuracy']
    b_5v_st  = report_stack_5v.loc[i, 'Balanced_Accuracy']
    
    cross_comp_rows.append({
        'Class': cls_lbl,
        'FedMML_Multiclass_BalAcc': f"{b_fed_mc*100:.2f}%",
        '5V_Ext_Multiclass_BalAcc': f"{b_5v_mc*100:.2f}%",
        'Delta_Multiclass_Generalization': f"{(b_5v_mc - b_fed_mc)*100:+.2f}%",
        'FedMML_Stacking_BalAcc': f"{b_fed_st*100:.2f}%",
        '5V_Ext_Stacking_BalAcc': f"{b_5v_st*100:.2f}%",
        'Delta_Stacking_Generalization': f"{(b_5v_st - b_fed_st)*100:+.2f}%"
    })

cross_comp_df = pd.DataFrame(cross_comp_rows)
print("=" * 120)
print("      CROSS-DATASET GENERALIZATION COMPARISON: FEDMML INTERNAL TEST vs EXTERNAL 5V DATASET")
print("=" * 120)
print(cross_comp_df.to_string(index=False))
print("=" * 120 + chr(10))

# Export CSV Reports
report_mc_5v.to_csv(os.path.join(reports_dir, 'external_5v_direct_multiclass_report.csv'), index=False)
report_stack_5v.to_csv(os.path.join(reports_dir, 'external_5v_hierarchical_stacking_report.csv'), index=False)
cross_comp_df.to_csv(os.path.join(reports_dir, 'fedmml_vs_5v_cross_dataset_comparison.csv'), index=False)
print(f"✓ External 5V reports exported successfully to {reports_dir}/")

In [ ]:
# ---------------------------------------------------------
# Step 7: Diagnostic Visualizations (Internal FedMML vs External 5V Benchmark)
# ---------------------------------------------------------
plots_dir = f"{ROOT}/plots"
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

esi_labels = [f"ESI {i}" for i in range(1, 6)]

# 1. 2x2 Normalized Confusion Matrices (FedMML Test vs 5V External)
fig, axes = plt.subplots(2, 2, figsize=(18, 15))

def render_cm(ax, y_true, y_pred, title_text, cmap='Blues'):
    cm = confusion_matrix(y_true, y_pred, labels=[1, 2, 3, 4, 5])
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    annot = np.empty_like(cm, dtype=object)
    for i in range(5):
        for j in range(5):
            annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.1f}%)"
    sns.heatmap(cm_norm, annot=annot, fmt='', cmap=cmap, cbar=True, ax=ax,
                vmin=0, vmax=1, xticklabels=esi_labels, yticklabels=esi_labels)
    ax.set_title(title_text, fontsize=11.5, fontweight='bold', pad=10)
    ax.set_xlabel("Predicted ESI Level", fontsize=10.5, fontweight='bold')
    ax.set_ylabel("True ESI Level", fontsize=10.5, fontweight='bold')

render_cm(axes[0, 0], y_test, preds_test_mc_fedmml, 
          f"[FedMML Test] Direct Multiclass LightGBM\nMacro Bal Acc: {report_mc_fedmml.loc[5, 'Balanced_Accuracy']*100:.2f}% | AUC: {report_mc_fedmml.loc[5, 'ROC_AUC']:.4f}", cmap='Blues')

render_cm(axes[0, 1], y_test, preds_test_stack_fedmml, 
          f"[FedMML Test] Hierarchical Stacking LightGBM\nMacro Bal Acc: {report_stack_fedmml.loc[5, 'Balanced_Accuracy']*100:.2f}% | AUC: {report_stack_fedmml.loc[5, 'ROC_AUC']:.4f}", cmap='Greens')

render_cm(axes[1, 0], y_5v, preds_5v_mc, 
          f"[External 5V] Direct Multiclass LightGBM\nMacro Bal Acc: {report_mc_5v.loc[5, 'Balanced_Accuracy']*100:.2f}% | AUC: {report_mc_5v.loc[5, 'ROC_AUC']:.4f}", cmap='Oranges')

render_cm(axes[1, 1], y_5v, preds_5v_stack, 
          f"[External 5V] Hierarchical Stacking LightGBM\nMacro Bal Acc: {report_stack_5v.loc[5, 'Balanced_Accuracy']*100:.2f}% | AUC: {report_stack_5v.loc[5, 'ROC_AUC']:.4f}", cmap='Purples')

plt.tight_layout()
cm_plot_path = os.path.join(plots_dir, "fedmml_vs_5v_confusion_matrix_grid.png")
plt.savefig(cm_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'fedmml_vs_5v_confusion_matrix_grid.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Confusion Matrix comparison grid saved to: {cm_plot_path}")

# 2. External 5V Multiclass ROC-AUC Curves
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

y_5v_bin = label_binarize(y_5v, classes=[1, 2, 3, 4, 5])
classes = [1, 2, 3, 4, 5]
n_classes = len(classes)
esi_colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))

def plot_roc_on_ax(ax, y_bin, probs, title_text):
    fpr, tpr, roc_aucs = dict(), dict(), dict()
    for i, cls in enumerate(classes):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], probs[:, i])
        roc_aucs[i] = auc(fpr[i], tpr[i])
    
    fpr["micro"], tpr["micro"], _ = roc_curve(y_bin.ravel(), probs.ravel())
    roc_aucs["micro"] = auc(fpr["micro"], tpr["micro"])
    
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_aucs["macro"] = auc(fpr["macro"], tpr["macro"])
    
    ax.plot(fpr["micro"], tpr["micro"], label=f"Micro-Average (AUC = {roc_aucs['micro']:.4f})", color='#e377c2', linestyle=':', linewidth=2.5)
    ax.plot(fpr["macro"], tpr["macro"], label=f"Macro-Average (AUC = {roc_aucs['macro']:.4f})", color='#17becf', linestyle='--', linewidth=2.5)
    
    for i in range(5):
        ax.plot(fpr[i], tpr[i], color=esi_colors[i], linewidth=2.0, label=f"ESI {i+1} (AUC = {roc_aucs[i]:.4f})")
    
    ax.plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Guess (AUC = 0.5000)')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11, fontweight='bold')
    ax.set_ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=11, fontweight='bold')
    ax.set_title(title_text, fontsize=12, fontweight='bold', pad=10)
    ax.legend(loc="lower right", fontsize=9.5, frameon=True, framealpha=0.95)
    ax.grid(True, linestyle='--', alpha=0.4)

plot_roc_on_ax(axes[0], y_5v_bin, probs_5v_mc, f"External 5V: Direct Multiclass LightGBM ROC (Macro AUC = {report_mc_5v.loc[5, 'ROC_AUC']:.4f})")
plot_roc_on_ax(axes[1], y_5v_bin, probs_5v_stack, f"External 5V: Hierarchical Stacking LightGBM ROC (Macro AUC = {report_stack_5v.loc[5, 'ROC_AUC']:.4f})")

plt.tight_layout()
roc_plot_path = os.path.join(plots_dir, "external_5v_roc_auc_curve.png")
plt.savefig(roc_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'external_5v_roc_auc_curve.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ External 5V ROC-AUC curves saved to: {roc_plot_path}")

In [ ]:
# ---------------------------------------------------------
# Step 8: Export Production Artifact Bundle & Metadata Manifest
# ---------------------------------------------------------
deploy_dir = f"{ROOT}/deploy"
os.makedirs(deploy_dir, exist_ok=True)

bundle = {
    'scaler': scaler,
    'feature_names': feature_names,
    'model_multiclass_lgbm': model_mc_lgbm,
    'hierarchical_stacking': {
        'l1_model': l1_model,
        'l2_model': l2_model,
        'l3a_model': l3a_model,
        'l3b_model': l3b_model,
        'meta_learner': meta_learner
    },
    'config_training': config['training']
}

bundle_file = os.path.join(deploy_dir, 'fedmml_lightgbm_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(bundle, f)

manifest = dict(
    pipeline='FedMML_LightGBM_ED_Triage_Classifier_With_5V_External_Benchmark',
    training_dataset='datasets/fedmml_ed_triage_dataset.csv',
    external_dataset='datasets/5v_cleandf.RData',
    config_file='config/triage_conf.json',
    config_training=config['training'],
    features=['age', 'sex', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2'],
    features_5v_mapping={
        'age': 'age',
        'sex': 'gender',
        'systolic_bp': 'triage_vital_sbp',
        'heart_rate': 'triage_vital_hr',
        'respiratory_rate': 'triage_vital_rr',
        'spo2': 'triage_vital_o2',
        'esi_level': 'esi'
    },
    n_classes=5,
    classes=['ESI 1', 'ESI 2', 'ESI 3', 'ESI 4', 'ESI 5'],
    fedmml_complete_cases=len(clean_df),
    ext_5v_complete_cases=len(mat_5v),
    fedmml_test_multiclass_metrics=report_mc_fedmml.to_dict(orient='records'),
    fedmml_test_stacking_metrics=report_stack_fedmml.to_dict(orient='records'),
    ext_5v_multiclass_metrics=report_mc_5v.to_dict(orient='records'),
    ext_5v_stacking_metrics=report_stack_5v.to_dict(orient='records'),
    generalization_comparison=cross_comp_rows
)

manifest_file = os.path.join(deploy_dir, 'fedmml_lightgbm_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Production Bundle   : {bundle_file}")
print(f"✓ Production Manifest : {manifest_file}")